# 实际利率变化信息增量

来源资格先行，固定单一增量。控制25价格特征，候选增加一个已发布实际利率20-session变化。年度内层成熟选择、H5与10/25bp费用不变；历史为复用诊断，不能称新留出。执行前已冻结110次拟合和12个原生账户场景，失败停止。

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE","true")
ENTRY_ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/"AGENTS.md").is_file())
ROOT=Path(os.environ.get("SOXL_EVIDENCE_ROOT",ENTRY_ROOT)).resolve()
SOURCE_ROOT=Path(os.environ.get("SOXL_RESEARCH_SOURCE_ROOT",ENTRY_ROOT)).resolve()
sys.path.insert(0,str(SOURCE_ROOT))
CARD=ENTRY_ROOT/"research"/"real_yield_information_increment_20261012"
PLAN=json.loads((CARD/"execution_plan.json").read_text())
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.workflow import R
from src.workflow.single_information_experiment import verify_information_execution_inputs,run_fixed_information_experiment
verify_information_execution_inputs(ROOT,CARD,PLAN,source_root=SOURCE_ROOT)
qlib.init(provider_uri=str(ROOT/PLAN["providers"]["price"]["provider_uri"]),region="us",kernels=1)
calendar=D.calendar(freq="day")
print({"price_provider_uri":PLAN["providers"]["price"]["provider_uri"],"source_provider_uri":PLAN["providers"]["information"]["provider_uri"],"region":"us","as_of":PLAN["as_of"],"native_market_calendar":len(calendar),"fixed_fit_budget":110,"fixed_native_cases":12})


In [ ]:
summary=run_fixed_information_experiment(root=ROOT,source_root=SOURCE_ROOT,card=CARD,execution_plan_path=CARD/"execution_plan.json")
print({k:summary[k] for k in ["recorder_id","fit_calls","native_cases","historical_gate_pass","diagnostic_gate_pass","incremental_information_pass","incremental_portfolio_pass","route_decision"]})


In [ ]:
from src.core.recorder_sources import load_verified_recorder_objects
objects,audit=load_verified_recorder_objects(CARD,expected_recorder_id=summary["recorder_id"],artifacts=["inputs/inputs.pkl","aggregate/tables.pkl"])
assert isinstance(objects["inputs/inputs.pkl"]["control"],DatasetH)
from src.data.information_increment import bind_matched_information_datasets
restored=bind_matched_information_datasets(objects["inputs/inputs.pkl"],as_of=PLAN["as_of"])
prepared=restored["control"].prepare("all",col_set=["feature","label","metadata"])
# Restoring native caches is a separate source-bound operation; the execution already used rebuilt native DatasetH.
tables=objects["aggregate/tables.pkl"]
display(tables["model_selection.csv"])
display(tables["backtest_benchmark_comparison.csv"])
display(tables["forecast_information.csv"])
display(tables["gates.csv"])
display(Markdown((CARD/"conclusion.md").read_text()))


中文结论执行后从conclusion.md同步。